# Layer A parity: ComplexTorch vs ComplexBox

Executable audit of Layer A numerical parity. ComplexBox commit `87b5e2cd9bba22ddd978bade6f614da7d6190db2` is the normative repository reference. The strict installed-repository test compares ComplexTorch, ComplexBox, and an independent SciPy/analytical reference whenever available.

For MVGC, `conditional=None` follows ComplexBox/MVGC and conditions on all remaining variables; `conditional=()` explicitly requests the unconditioned marginal question.

In [1]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'validation' / 'layer_a').exists():
    ROOT = Path('/mnt/data/ctwork/complextorch-main')
OUT = ROOT / 'validation' / 'layer_a' / 'results_notebook'
OUT.mkdir(parents=True, exist_ok=True)
print('repo:', ROOT)

repo: /mnt/data/ctwork/complextorch-main


## Run the full Layer A suite

The subprocess output is captured to keep the notebook readable. A non-zero return code fails this cell.

In [2]:
proc = subprocess.run([sys.executable, str(ROOT/'validation/layer_a/run_all.py'), '--output', str(OUT)], cwd=ROOT, text=True, capture_output=True)
print(proc.stdout.strip().splitlines()[-1])
if proc.returncode != 0:
    print(proc.stderr[-4000:])
    raise RuntimeError(f'Layer A failed with return code {proc.returncode}')

Layer A summary: 205 checks, 0 classified failures, 0 non-zero scripts.


## Aggregated result

In [3]:
summary = json.loads((OUT/'layer_a_summary.json').read_text())
counts = {}
for row in summary:
    counts[row['status']] = counts.get(row['status'], 0) + 1
print('checks:', len(summary))
print('status counts:', counts)
failures = [r for r in summary if r['status'] == 'FAIL']
print('failures:', len(failures))

checks: 205
status counts: {'PASS_NUMERICAL': 165, 'PASS_EXACT': 40}
failures: 0


## Strict installed-repository triplets

In [4]:
triplets = json.loads((OUT/'installed_repo_triplets_layera.json').read_text())
print('triplet checks:', len(triplets))
print('triplet failures:', sum(r['status']=='FAIL' for r in triplets))
worst = sorted(triplets, key=lambda r: r.get('absolute_error') or 0, reverse=True)[:8]
for r in worst:
    print(f"{r['check']}: error={r['absolute_error']:.3e}, tol={r['tolerance']:.3e}")

triplet checks: 85
triplet failures: 0
ssm.generalized_dare.P::complextorch_vs_complexbox: error=1.941e-11, tol=5.000e-09
ssm.generalized_dare.P::complextorch_vs_reference: error=1.941e-11, tol=5.000e-09
ssm.generalized_dare.ct_residual::residual_bound: error=1.167e-11, tol=5.000e-09
mvgc.default_conditional.temporal.ct_vs_complexbox::complextorch_default_vs_complexbox_default: error=1.857e-14, tol=3.000e-10
var3.lyapunov.direct::complextorch_vs_complexbox: error=7.772e-15, tol=3.000e-10
var3.lyapunov.doubling::complextorch_vs_complexbox: error=7.772e-15, tol=3.000e-10
var3.lyapunov.direct::complexbox_vs_reference: error=7.550e-15, tol=3.000e-10
var3.lyapunov.doubling::complexbox_vs_reference: error=7.550e-15, tol=3.000e-10


## MVGC default-conditioning parity

In [5]:
mvgc = [r for r in triplets if 'mvgc.default_conditional' in r['check']]
for r in mvgc:
    print(r['check'], r['status'], f"error={r['absolute_error']:.3e}")
assert mvgc and all(r['status'] != 'FAIL' for r in mvgc)

mvgc.default_conditional.temporal.default_vs_explicit_remaining::complextorch_default_vs_complextorch_explicit_remaining PASS_NUMERICAL error=0.000e+00
mvgc.default_conditional.temporal.ct_vs_complexbox::complextorch_default_vs_complexbox_default PASS_NUMERICAL error=1.857e-14
mvgc.default_conditional.spectral.default_vs_explicit_remaining::complextorch_default_vs_complextorch_explicit_remaining PASS_NUMERICAL error=0.000e+00
mvgc.default_conditional.spectral.ct_vs_complexbox::complextorch_default_vs_complexbox_default PASS_NUMERICAL error=5.178e-16


## Interpretation

Passing this notebook means the checked Layer A primitives agree with ComplexBox to their declared numerical tolerances, with independent references where available. It does not turn stress-test tolerances into mathematical identities; the JSON diagnostics retain the actual residuals and errors.